In [1]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

c:\Users\Camille\Documents\TWR\deep_agents_twr


In [2]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
from app.config.container import campaign_service, request_service

source = "outbrain"
emb_config = "fasttext"

# hashes = await campaign_service.fetch_recent_active_campaigns(
#       traffic_source=source,
#       limit=10
# )
hashes = ["9xvp496ldw"]

print(hashes)
print(len(hashes))

requests = await request_service.fetch_training_sample_by_hashes(
      hashes=hashes,
      limit_each=1000,
      only_rule_id=True
)

data = pd.DataFrame(requests)
# print(data["rule_id_list"].value_counts())

c:\Users\Camille\Documents\TWR\deep_agents_twr\.venv\Lib\site-packages\motor\core.py:171: UserWarning: You appear to be connected to a DocumentDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/documentdb
  delegate = self.__delegate_class__(*args, **kwargs)


Garantindo índices...
['9xvp496ldw']
1


In [ ]:
import pandas as pd
from app.utils.analise import parse_dict_col

data['headers'] = data['headers'].apply(parse_dict_col)
data['request'] = data['request'].apply(parse_dict_col)

def dict_to_tokens(d):
    tokens = []
    for k, v in d.items():
        tokens.append(f"{k}:{v}")
    return tokens


In [27]:
def exctract_user_agent(headers):
      ua = headers.get("user-agent")

      return ua

data["ua"] = data["headers"].apply(exctract_user_agent)

In [ ]:
from sentence_transformers import SentenceTransformer

from app.services.request_clustering_service import RequestClusteringPipeline

model = SentenceTransformer('all-MiniLM-L6-v2') 

pipeline = RequestClusteringPipeline(
    embedding_model=model,
    max_header_freq=0.7,
    min_cluster_size=30
)

df_result = pipeline.run(data)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4133.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2. Serializando requisições e gerando embeddings...
1. Calculando cardinalidade e filtrando headers...
   -> Headers mantidos (55): {'sec-fetch-site', 'cf-metro-code', 'accept', 'cookie2', 'sec-ch-ua-platform', 'x-threat-score', 'cf-timezone', 'x-http-tls-version', 'upgrade-insecure-requests', 'x-forwarded-host', 'cf-cert-verified', 'sec-fetch-user', 'save-data', 'origin', 'x-scheme', 'sec-fetch-dest', 'cf-cert-revoked', 'sec-gpc', 'sec-fetch-mode', 'x-requested-with', 'cache-control', 'cf-cert-presented', 'dnt', 'accept-language', 'content-length', 'host', 'cf-ipcountry', 'from', 'x-http-version', 'x-forwarded-scheme', 'user-agent', 'accept-encoding', 'forwarded', 'x-http-ssl-protocol', 'x-http-method', 'cf-iplongitude', 'cf-postal-code', 'sec-ch-ua', 'if-modified-since', 'x-verified-bot-category', 'cf-iplatitude', 'sec-ch-ua-mobile', 'x-client-bot', 'cdn-loop', 'content-type', 'cf-ipcontinent', 'cf-visitor', 'cf-region', 'cf-ipcity', 'x-asn', 'x-forwarded-proto', 'cf-region-code', 'x

Batches: 100%|██████████| 32/32 [00:33<00:00,  1.06s/it]


   -> Gerando embeddings FILTERED...


Batches: 100%|██████████| 32/32 [00:35<00:00,  1.11s/it]


3. Aplicando t-SNE (Redução de dimensionalidade)...
4. Clusterizando com HDBSCAN e classificando...
5. Gerando visualizações 3D...


Pipeline finalizado com sucesso!


In [36]:
df_result["classification"].value_counts()

classification
correct_bot        1000
correct_unsafe      849
unsafe_pred_bot     151
Name: count, dtype: int64